results of each tracking are in video files

In [1]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
import tensorflow as tf
import time
import os
import json
import kagglehub
from djitellopy import Tello
from datetime import datetime
from ultralytics import YOLO
from collections import defaultdict

path = kagglehub.model_download("tensorflow/ssd-mobilenet-v2/tensorFlow2/fpnlite-320x320")

# Load pre-trained MobileNet-SSD model (first)
modelCamera = tf.saved_model.load("saved_model")
modelDrone = YOLO("YOLO/yolov8s.pt")

# Function to perform human detection and distance estimation
def detect_and_estimate_distance(frame, threshold_area, model=modelCamera):
    input_tensor = tf.convert_to_tensor(frame)
    input_tensor = input_tensor[tf.newaxis,...]
    
    detections = model(input_tensor)

    boxes = detections['detection_boxes'][0].numpy()
    classes = detections['detection_classes'][0].numpy().astype(np.int32)
    scores = detections['detection_scores'][0].numpy()

    frame_height, frame_width, _ = frame.shape

    for i in range(len(scores)):
        if scores[i] > 0.5 and classes[i] == 1:  # Class 1 corresponds to 'person'
            ymin, xmin, ymax, xmax = boxes[i]
            (left, right, top, bottom) = (xmin * frame_width, xmax * frame_width, ymin * frame_height, ymax * frame_height)
            area = (right - left) * (bottom - top)
            
            # Draw bounding box
            cv2.rectangle(frame, (int(left), int(top)), (int(right), int(bottom)), (0, 255, 0), 2)
            
            # Check if area exceeds threshold
            if area > threshold_area:
                return True, frame
    
    return False, frame


def detect_objects_in_video(video_path, output_video_path, model=modelDrone):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    object_counts = defaultdict(int)

    # Get frame width and height
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))

    # Define the codec and create VideoWriter object for the output video
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_video_path, fourcc, 30.0, (frame_width, frame_height))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detect objects in the frame
        results = model(frame)

        for result in results:
            for box in result.boxes:
                class_id = int(box.cls[0])
                label = model.names[class_id]
                object_counts[label] += 1

                # Draw bounding boxes and labels on the frame
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                confidence = box.conf[0]
                color = (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"{label} {confidence:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Write the processed frame to the output video
        out.write(frame)
        frame_count += 1

    cap.release()
    out.release()
    return object_counts, frame_count

def calculate_percentage(object_counts, frame_count):
    percentages = {obj: (count / frame_count) * 100 for obj, count in object_counts.items()}
    return percentages

def save_json(data, output_folder):
    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)
    
    # Define the output JSON file path
    json_file_path = os.path.join(output_folder, "object_detection_results.json")

    # Save data as JSON
    with open(json_file_path, "w") as json_file:
        json.dump(data, json_file, indent=4)


def run_video_object_detection(video_path, output_folder, model=modelDrone):

    # Output video path with object detection
    output_video_path = os.path.join(output_folder, 'tello_video_detected.avi')

    # Perform object detection and count objects
    object_counts, frame_count = detect_objects_in_video(video_path, model, output_video_path)
    
    # Calculate percentages
    object_percentages = calculate_percentage(object_counts, frame_count)

    # Prepare data for JSON
    data = {obj: f"{percentage:.2f}%" for obj, percentage in object_percentages.items()}

    # Save the results to a JSON file
    save_json(data, output_folder)
    
    print(f"Object detection results and video saved to {output_folder}/")


def tello_move(tello):
    # Take off
    tello.takeoff()
    time.sleep(1)

    # Move up by 1 meter
    tello.move_up(100)
    time.sleep(1)

    # Move forward by 1 meter
    tello.move_forward(100)
    time.sleep(1)

    # Move back by 1 meter
    tello.move_back(100)
    time.sleep(1)

    # Land
    tello.land()

def list_connected_devices(max_devices=10):
    available_devices = []
    for device_index in range(max_devices):
        cap = cv2.VideoCapture(device_index)
        if cap.isOpened():
            available_devices.append(device_index)
            cap.release()
    return available_devices
#print("Connected video devices:", list_connected_devices())

def drone_init():

    # Create a Tello object
    tello = Tello()

    # Connect to the Tello drone
    tello.connect()

    # Print the battery level
    battery = tello.get_battery()
    print(f"Battery level: {battery}%")

    # Get current date for folder name
    current_date = datetime.now().strftime("%Y-%m-%d")
    if not os.path.exists(f'{current_date}'):
        os.makedirs(f'{current_date}')

    # Start video stream
    tello.streamon()

    # Initialize the video writer
    frame_read = tello.get_frame_read()
    time.sleep(2)
    frame = frame_read.frame
    height, width, _ = frame.shape
    video = cv2.VideoWriter(f'{current_date}/drone.avi', cv2.VideoWriter_fourcc('M','J','P','G'), 30.0, (width, height))
    
    print("Starting video capture. Press 'q' to quit.")

    # Move the drone
    #tello_move(tello)

    while True:
        # Get the current frame from the drone
        frame = frame_read.frame
        
        # Write the frame to the video file
        video.write(frame)
        
        # Display the frame (optional)
        cv2.imshow("Tello Video Stream", frame)
        
        # Check for 'q' key press to quit early
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        # Stop recording when the drone lands
        #if not tello.is_flying:
            #break

    video.release()  
    cv2.destroyAllWindows() 

    # Stop video stream
    tello.streamoff()

def main(camera):
    # Access webcam and process frames
    cap = cv2.VideoCapture(camera)
    threshold_area = 50000  # Define an appropriate threshold area

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        is_close, processed_frame = detect_and_estimate_distance(frame, threshold_area, model=modelCamera)

        if is_close:
            # Run drone.py
            cv2.imshow("Frame", processed_frame)
            # Save frame to folder with name of current date
            current_date = datetime.now().strftime("%Y-%m-%d")
            if not os.path.exists(f'{current_date}'):
                os.makedirs(f'{current_date}')
            frame_path = os.path.join(f'{current_date}', f"frame_{datetime.now().strftime('%H-%M-%S')}.jpg")
            cv2.imwrite(frame_path, processed_frame)

            # Initialize the drone
            drone_init()

            break
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()



2024-07-11 18:41:40.197979: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
sh: python: command not found
